# Part 3: Explainable AI Module (SHAP)

Loads the model trained in Part 2 and adds the explainability layer -- global
feature importance and per-candidate SHAP-based explanations. This is the
"Explainable AI Module" block from the proposed architecture.

**Input:** `engineered_features.csv`, `rf_model.joblib`, `model_config.joblib`
(from Parts 1-2)
**Output:** none saved -- this notebook produces analysis/plots. The per-candidate
explanation function is reused directly in Part 4.

## 1. Setup

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()  # upload engineered_features.csv, rf_model.joblib, model_config.joblib
except ImportError:
    pass

!pip install shap scikit-learn -q


In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import shap

feature_df = pd.read_csv("engineered_features.csv")
final_model = joblib.load("rf_model.joblib")
config = joblib.load("model_config.joblib")

FEATURES = config["features"]
EXPERIENCE_MEDIAN = config["experience_median"]
CONFIDENCE_THRESHOLD = config["confidence_threshold"]

print("Model classes:", final_model.classes_)
print("Features:", FEATURES)


In [ ]:
def impute_experience_score(scores, fill_value):
    return scores.fillna(fill_value)

X_full = feature_df[FEATURES].copy()
X_full["experience_match_score"] = impute_experience_score(X_full["experience_match_score"], EXPERIENCE_MEDIAN)


## 2. Global feature importance

`shap.TreeExplainer` is exact (not approximate) for tree models like ours -- a key
reason Random Forest was chosen over other model families.

In [ ]:
explainer = shap.TreeExplainer(final_model)
shap_values_all = explainer.shap_values(X_full)

shap.summary_plot(shap_values_all[:, :, list(final_model.classes_).index("match")],
                   X_full, plot_type="bar", show=False)
plt.title("Feature importance for predicting Match")
plt.tight_layout()
plt.show()


In [ ]:
shap.summary_plot(shap_values_all[:, :, list(final_model.classes_).index("partial match")],
                   X_full, plot_type="bar", show=False)
plt.title("Feature importance for predicting Partial Match")
plt.tight_layout()
plt.show()


## 3. Per-candidate explanation

Turns SHAP's numeric output into a natural-language explanation for a single
candidate -- matches the "Reasons" format from the project proposal.

In [ ]:
FEATURE_LABELS = {
    "skill_match_score": "Skill Match",
    "matched_skill_count": "Number of Matched Skills",
    "missing_skill_count": "Number of Missing Skills",
    "experience_match_score": "Experience Match",
    "resume_experience_mentioned": "Resume States Experience",
    "jd_experience_mentioned": "JD States Required Experience",
    "semantic_similarity_score": "Overall Semantic Fit",
    "domain_match": "Domain Match",
}


def explain_prediction(X_row: pd.DataFrame, model=final_model, explainer=explainer,
                        confidence_threshold: float = CONFIDENCE_THRESHOLD) -> dict:
    """
    X_row: a single-row DataFrame with the FEATURES columns.
    Returns prediction, confidence, human-review flag, and top contributing factors.
    This function is reused directly in Part 4.
    """
    proba = model.predict_proba(X_row)[0]
    pred_class = model.classes_[np.argmax(proba)]
    confidence = float(np.max(proba))
    needs_review = confidence < confidence_threshold

    sv = explainer.shap_values(X_row)
    class_idx = list(model.classes_).index(pred_class)
    contributions = sv[0, :, class_idx]
    impact = sorted(zip(FEATURES, contributions), key=lambda t: -abs(t[1]))

    reasons = [{
        "factor": FEATURE_LABELS[feat],
        "value": round(float(X_row[feat].values[0]), 2),
        "direction": "supported" if contrib > 0 else "worked against",
        "impact": round(float(contrib), 3),
    } for feat, contrib in impact[:4]]

    return {
        "prediction": pred_class,
        "confidence": round(confidence, 2),
        "needs_human_review": needs_review,
        "reasons": reasons,
    }


## 4. Test on a few real candidates from the dataset

In [ ]:
for idx in [0, 5, 50]:
    row = X_full.iloc[[idx]]
    result = explain_prediction(row)
    print(f"--- Row {idx} (true label: {feature_df.iloc[idx]['match_label']}) ---")
    print(f"Prediction: {result['prediction'].upper()}  (confidence: {result['confidence']:.0%})")
    if result["needs_human_review"]:
        print("Flagged for human review")
    for r in result["reasons"]:
        print(f"  - {r['factor']} ({r['value']}) {r['direction']} (impact: {r['impact']:+.3f})")
    print()
